## Setup

IMPORTANT: in case files in /home/ubuntu/Project/libero_development/src/ are changed, must also submit them to workers (to avoid reading from an old file on WORKER'S disk): from head VM's terminal, run:

```bash
SRC_DIR="/home/ubuntu/Project/libero_development/src"
DEST_DIR="/home/ubuntu/Project/libero_development/src"

WORKER_IPS=(
    "10.67.22.254" "10.67.22.34" "10.67.22.145" "10.67.22.121"
    "10.67.22.192" "10.67.22.18" "10.67.22.187" "10.67.22.48"
)

for ip in "${WORKER_IPS[@]}"; do
    echo "Syncing src/ to $ip..."
    ssh -o StrictHostKeyChecking=no ubuntu@"$ip" "mkdir -p $DEST_DIR"
    rsync -avz --exclude '__pycache__/' "$SRC_DIR/" ubuntu@"$ip":"$DEST_DIR/"
done
```


IMPORTANT: to kill existing Dask processes (from bash):
```bash
for ip in 10.67.22.194 10.67.22.254 10.67.22.34 10.67.22.145 10.67.22.121 10.67.22.192 10.67.22.18 10.67.22.187 10.67.22.48; do
    echo "Killing on $ip..."
    ssh -o StrictHostKeyChecking=no ubuntu@"$ip" "pkill -9 -f dask-scheduler; pkill -9 -f dask-worker; pkill -9 -f dask_ssh" 
done
```

USEFUL: to combine different csv in a unique csv:
```bash
head -n 1 file.csv > combined.csv
for file in *i_tuoi_file*.csv; do tail -n +2 "$file" >> combined.csv; done
```

USEFUL: to check if a certain port occupied prevents strating worker(s):
```bash
ssh ubuntu@10.67.22.194 "lsof -i :8786"
```
then to kill eg this PID:
```bash
ssh ubuntu@10.67.22.194 "fuser -k 8786/tcp"
```


In [1]:
import numpy as np
import pandas as pd
from src.kmeans_parallel import kmeans_parallel
from src.data_loader import load_dataset
from src.benchmark import run_single_test, run_benchmark, calculate_inertia, combinations_fn
from src.launch_cluster import launch_cluster, shutdown_cluster
import time

In [2]:
# --- Cluster ---
N_WORKERS = 8      # tra 1 e 8 (nodi disponibili in launch_cluster.py)
NUM_PARTITIONS = 8 * N_WORKERS   # regola empirica: >= n_threads_per_worker * n_workers
# --- Algoritmo k-means|| ---
#K = 500                # numero di cluster finali
#L = 250                # oversampling factor (assoluto). In alternativa: L = round(L_OVER_K * K)
#R = 10                  # numero di round dell'inizializzazione parallela
#MAX_ITER_FIT = 100      # iterazioni massime della fase di Lloyd's (fit)

SEED = 42

In [3]:
# --- Dataset ---
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
# link used by sklearn function fetch_kddcup99
#***for 10% dataset***

DATASET_URL_FULL="https://ndownloader.figshare.com/files/5976045"
#***FULL DATASET***

RAW_GZ_PATH_10PC   = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz" # compressed (.gz) dataset file
PARQUET_PATH_10PC = '/tmp/kddcup_data_shards' # directory of Parquet shard files on master (one per partition)

RAW_GZ_PATH_FULL = "/home/ubuntu/backup/libero_development/data/kddcup_data_full.gz"
PARQUET_PATH_FULL = '/tmp/kddcup_data_full_shards'

# column names of KDD dataset, from source code of the above sklearn function;
# "protocol_type","service","flag" are non-numeric so they will be dropped later,
# as will be "label" and the constant column 'num_outbound_cmds' 

COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label"
]

In [4]:
# DO NOT RUN if already started! :
cluster, client = launch_cluster(N_WORKERS)

Initializing the SSH cluster with 8 workers...
Selected workers: ['10.67.22.254', '10.67.22.34', '10.67.22.145', '10.67.22.121', '10.67.22.192', '10.67.22.18', '10.67.22.187', '10.67.22.48']


2026-09-12 18:10:22,418 - distributed.deploy.ssh - INFO - 2026-09-12 18:10:22,417 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-09-12 18:10:22,461 - distributed.deploy.ssh - INFO - 2026-09-12 18:10:22,460 - distributed.scheduler - INFO - State start
2026-09-12 18:10:22,465 - distributed.deploy.ssh - INFO - 2026-09-12 18:10:22,465 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.67.22.194:8786
2026-09-12 18:10:24,326 - distributed.deploy.ssh - INFO - 2026-09-12 18:10:24,324 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.145:44843'
2026-09-12 18:10:24,409 - distributed.deploy.ssh - INFO - 2026-09-12 18:10:24,408 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.18:36903'
2026-09-12 18:10:24,420 - distributed.deploy.ssh - INFO - 2026-09-12 18:10:24,420 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.48:

Cluster started and connection established successfully!



#### IF INSTEAD THE CLUSTER HAS ALREADY BEEN STARTED:

In [6]:
from dask.distributed import Client

SCHEDULER_ADDRESS = "tcp://10.67.22.194:8786"

try:
    client = Client(SCHEDULER_ADDRESS, timeout="10s")
    print("Connected to cluster successfully!")
    print(f"Dask Dashboard link: {client.dashboard_link}")

except Exception as e:
    print(f"Connection error: {e}")

Connected to cluster successfully!
Dask Dashboard link: http://10.67.22.194:8787/status


2026-09-11 12:54:05,391 - distributed.client - ERROR - Failed to reconnect to scheduler after 10.00 seconds, closing client


## Load dataset

### 10%

In [7]:
#10 percent:

start=time.time()
X_bag_10_percent, (mean_ar, std_ar)  = load_dataset(n_partitions=NUM_PARTITIONS,
                   client=client,
                   dataset_url=DATASET_URL_10PC,
                   raw_gz_path=RAW_GZ_PATH_10PC,
                   parquet_path=PARQUET_PATH_10PC,
                   col_names=COL_NAMES,
                   force_download=True)
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

Converting .gz -> Parquet shards...
Parquet shards created (64 files, snappy).
Constant columns: ['num_outbound_cmds', 'is_host_login']
Distributed dask.array created with 64 partitions.
Number of samples: 494021
Time elapsed: 6.24 s


#### FULL

In [6]:
# full:

start=time.time()
X_bag_full, (mean_ar, std_ar)=load_dataset(n_partitions=NUM_PARTITIONS,
                   client=client,
                   dataset_url=DATASET_URL_FULL,
                   raw_gz_path=RAW_GZ_PATH_FULL,
                   parquet_path=PARQUET_PATH_FULL,
                   col_names=COL_NAMES,
                   force_download=True)
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

Converting .gz -> Parquet shards...
Parquet shards created (64 files, snappy).
Constant columns: ['num_outbound_cmds']
Distributed dask.array created with 64 partitions.
Number of samples: 4898431
Time elapsed: 39.75 s


## Run experiments

### Varying number of L/K

#### k=500,1000

In [14]:
combos = [                      
    #(N_WORKERS, NUM_PARTITIONS, l_over_k, R),   # under-partitioned
    (N_WORKERS, NUM_PARTITIONS, 0.1, 15),
    (N_WORKERS, NUM_PARTITIONS, 0.5, 5),   
    (N_WORKERS, NUM_PARTITIONS, 1, 5),   
    (N_WORKERS, NUM_PARTITIONS, 2, 5),   
    (N_WORKERS, NUM_PARTITIONS, 10, 5),   
]
K_VALUES=[500,1000]
MAX_ITER_FIT=80 
avg_iters=10

In [ ]:
dataset_bag = X_bag_full
df = run_benchmark(client, X_bag=dataset_bag, combinations=combos,
                   k_values=K_VALUES, label="l_su_k_final",
                   max_iter_fit=MAX_ITER_FIT, seed=SEED,
                   averaging_iterations=avg_iters,
                   policy='fixed')

Testing: k=500, workers=8, partitions=64, l=50 (l/k=0.1), r=15
 Iterating 10 times.
Doing averaging iteration number 0...
Stopped at LLoyd's iteration 9 due to relative tolerance threshold 0.0001
 -> Final cost: 587169.17 | Time: 44.82s | Time for LLoyd iterations only: 25.03s
Doing averaging iteration number 1...
Stopped at LLoyd's iteration 11 due to relative tolerance threshold 0.0001
 -> Final cost: 588707.55 | Time: 47.10s | Time for LLoyd iterations only: 28.23s
Doing averaging iteration number 2...
Stopped at LLoyd's iteration 14 due to relative tolerance threshold 0.0001
 -> Final cost: 580004.63 | Time: 579.84s | Time for LLoyd iterations only: 342.80s
Doing averaging iteration number 3...
Stopped at LLoyd's iteration 15 due to relative tolerance threshold 0.0001
 -> Final cost: 585842.68 | Time: 388.40s | Time for LLoyd iterations only: 32.13s
Doing averaging iteration number 4...
Stopped at LLoyd's iteration 11 due to relative tolerance threshold 0.0001
 -> Final cost: 59108

### Run with random (tabs 3,4)

In [14]:
combos = [
    (N_WORKERS, NUM_PARTITIONS, 0, 0)
    # R=0 per avere inizializzazione random (l can have any value, as it is not used)
]
avg_iters = 10
K_VALUES = [500, 1000]
MAX_ITER_FIT = 20 # as in the paper

In [15]:
dataset_bag = X_bag_full
df = run_benchmark(client, X_bag=dataset_bag, combinations=combos,
                   k_values=K_VALUES, label="random_l_su_k_final",
                   max_iter_fit=MAX_ITER_FIT, seed=SEED, averaging_iterations=avg_iters,policy='fixed')

Testing: k=500, workers=8, partitions=64, l=1 (l/k=0), r=0
 Iterating 10 times.
Doing averaging iteration number 0...


/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 255 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(


 -> Final cost: 31462276.63 | Time: 48.86s | Time for LLoyd iterations only: 35.88s
Doing averaging iteration number 1...


/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 244 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(


 -> Final cost: 30871489.57 | Time: 50.19s | Time for LLoyd iterations only: 37.41s
Doing averaging iteration number 2...


...

 -> Final cost: 19689814.91 | Time: 73.74s | Time for LLoyd iterations only: 59.78s
Doing averaging iteration number 8...


/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 585 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(


 -> Final cost: 29058133.66 | Time: 72.06s | Time for LLoyd iterations only: 58.27s
Doing averaging iteration number 9...


/home/ubuntu/Project/libero_development/src/kmeans_parallel.py:756: UserWarning: 545 cluster vuoti dopo l'assegnazione: i centroidi corrispondenti restano al valore dell'iterazione precedente
  warnings.warn(


 -> Final cost: 14117111.67 | Time: 73.41s | Time for LLoyd iterations only: 59.89s

Risultati completi salvati in: /home/ubuntu/Project/libero_development/results/random_l_su_k_final_20260911125535.csv
--- Benchmark Complete ---
{'k': 1000, 'l': 1, 'r': 0, 'r_effective': 0, 'partitions': 64, 'initial_cost': 72102837.91192953, 'final_cost': 14117111.665787142, 'time': 73.41301369667053, 'lloyd_time': 59.890403270721436, 'num effective lloyd iters': 20, 'seed': 51, 'cost_history': [72102837.91192953, 66055503.157968424, 50187080.53918832, 32515582.183374505, 28027813.34190529, 24956465.543666575, 24591913.800223757, 24484660.144896287, 24060604.394411057, 23621540.3310806, 23308980.573266115, 23106241.80921869, 18040638.374009863, 17900971.635117877, 17692638.96737588, 17342005.154646434, 17073725.897499997, 16630714.157862924, 14941539.437472705, 14119873.261701064], 'iter_times': [15.60570240020752, 2.60931134223938, 2.2835216522216797, 2.2858388423919678, 2.3232619762420654, 2.233813

In [ ]:
df = run_table34(client=client, X_bag_full=X_bag_full, l_over_k_values=(1,2),include_random=True, n_runs=5)

## Cluster shutdown

In [8]:
shutdown_cluster(cluster, client)

Cluster and client closed.
